# Developing a function to align DIC And DAPI channels

Too much manual finetuning of filters. Abandoned.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = search_path = (
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET161_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET155_ASH1Q670_04_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET147_ECE1Q670_06_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_07_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET145_ASH1Q670_02_CY5, DAPI",
    )
file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC*.tif",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit",
        "correct_DIC_shift": [10,0]
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
dapi_files = []
dic_files = []
for p in search_path: 
    dapi_files.append(find_files_by_pattern(p, file_pattern[2], verbose=True))
    dic_files.append(find_files_by_pattern(p, file_pattern[3], verbose=True))

dapi_imgs = [ImageContainer(img,config) for img in dapi_files]
dic_imgs = [ImageContainer(img,config) for img in dic_files]
merged_imgs = [dapi+dic for dapi,dic in zip(dapi_imgs,dic_imgs)]

In [ ]:
import cv2
import numpy as np
from skimage.registration import phase_cross_correlation
from skimage.filters import sobel
from image_processing_tools.image_class.image_filters import ImageJProcessor

print(dapi_imgs[2].name)
ref_img = dapi_imgs[2].merge()
ref_img_filters = ImageJProcessor(ref_img)
ref_img_filters.gaussian_blur()
ref_img_filters.fft_bandpass_filter()
ref_img_filters.frangi_imagej_ops()
ref_img = ref_img_filters.return_image()

print(dic_imgs[2].name)
mov_img = dic_imgs[2].merge()
mov_img_filters = ImageJProcessor(mov_img)
mov_img_filters.gaussian_blur()
mov_img_filters.enhance_contrast_clahe()
mov_img_filters.fft_bandpass_filter()
mov_img_filters.imagej_rolling_ball()
mov_img = mov_img_filters.return_image()

ref_edges = sobel(ref_img.astype(np.float32) / ref_img.max())
mov_edges = sobel(mov_img.astype(np.float32) / mov_img.max())

shift, error, diffphase = phase_cross_correlation(
        ref_edges, 
        mov_edges, 
        upsample_factor=10  # Sub-pixel accuracy down to 1/10th of a pixel
    )
    
# 4. Convert to integers since scipy.ndimage.shift expects whole numbers
rounded_shift = [int(round(shift[0])), int(round(shift[1]))]

print(f"Calculated shift: {rounded_shift} (Error: {error:.3f})")

In [ ]:
import matplotlib.pyplot as plt
from image_processing_tools.image_class.image_filters import ImageJProcessor

ref_img = dapi_imgs[2].merge()
ref_img_filters = ImageJProcessor(ref_img)
ref_img_filters.gaussian_blur()
# ref_img_filters.fft_bandpass_filter()
ref_img_filters.frangi_imagej_ops()
ref_img = ref_img_filters.return_image()

mov_img = dic_imgs[2].merge()
mov_img_filters = ImageJProcessor(mov_img)
mov_img_filters.gaussian_blur()
mov_img_filters.enhance_contrast_clahe()
mov_img_filters.fft_bandpass_filter()
mov_img_filters.frangi_imagej_ops()
mov_img = mov_img_filters.return_image()

ref_edges = sobel(ref_img.astype(np.float32) / ref_img.max())
mov_edges = sobel(mov_img.astype(np.float32) / mov_img.max())

In [ ]:
import cv2
import numpy as np
from skimage.registration import phase_cross_correlation
from skimage.filters import sobel

def calculate_dynamic_shift(reference_path, moving_path):
    """
    Calculates the Y, X shift required to align a moving image to a reference image.
    Uses Sobel filtering to improve multi-modal (DAPI vs DIC) registration.
    """
    # 1. Load both images
    ref_img = cv2.imread(str(reference_path), cv2.IMREAD_UNCHANGED)
    mov_img = cv2.imread(str(moving_path), cv2.IMREAD_UNCHANGED)
    
    if ref_img is None or mov_img is None:
        raise ValueError("Could not load one or both images for alignment.")

    # 2. Normalize and apply edge detection to isolate structural features
    ref_edges = sobel(ref_img.astype(np.float32) / ref_img.max())
    mov_edges = sobel(mov_img.astype(np.float32) / mov_img.max())

    # 3. Calculate the shift using phase cross-correlation
    # shift will be a tuple: (y_shift, x_shift)
    shift, error, diffphase = phase_cross_correlation(
        ref_edges, 
        mov_edges, 
        upsample_factor=10  # Sub-pixel accuracy down to 1/10th of a pixel
    )
    
    # 4. Convert to integers since scipy.ndimage.shift expects whole numbers
    rounded_shift = [int(round(shift[0])), int(round(shift[1]))]
    
    print(f"Calculated shift for {moving_path.name}: {rounded_shift} (Error: {error:.3f})")
    return rounded_shift
